# 01 Data Setup

Loads the raw CSV files in `data/` into DuckDB and verifies the load. Raw layer only: no cleaning or business rules.

| Table | Grain | Key |
| --- | --- | --- |
| `train_run_events` | One row per train run per station stop | `service_id`, `station_sequence` |
| `train_master` | One row per train | `train_id` |
| `station_master` | One row per station | `station_id` |

Idempotent full refresh. Source files are read-only. The data is synthetic.

## 1. Setup

Paths resolve from the project root. Missing source files fail here.

In [1]:
import duckdb

from pathlib import Path

# project root is the folder that holds data/, whether the notebook runs from the root or from notebooks/
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
DATA_DIR = PROJECT_ROOT / "data"
DB_PATH = PROJECT_ROOT / "railpulse.duckdb"
RAW_TABLES = ["train_run_events", "train_master", "station_master"]

# check the source files are there before loading anything
for table in RAW_TABLES:
    csv_path = DATA_DIR / f"{table}.csv"
    assert csv_path.is_file(), f"Missing file: {csv_path}"
    print(f"Found {csv_path.relative_to(PROJECT_ROOT)}")

print(f"DuckDB version: {duckdb.__version__}")

Found data/train_run_events.csv
Found data/train_master.csv
Found data/station_master.csv
DuckDB version: 1.5.6


## 2. Connect to DuckDB

`railpulse.duckdb` is the hand-off to downstream notebooks. It is a build artefact and can be rebuilt at any time.

In [2]:
con = duckdb.connect(str(DB_PATH))
print(f"Connected to {DB_PATH.relative_to(PROJECT_ROOT)}")

Connected to railpulse.duckdb


## 3. Create tables

Full refresh with `CREATE OR REPLACE TABLE`. Types are inferred by `read_csv` and reviewed in section 4.

### 3.1 `train_run_events`

In [3]:
con.execute(f"""
    CREATE OR REPLACE TABLE train_run_events AS
    SELECT *
    FROM read_csv('{(DATA_DIR / "train_run_events.csv").as_posix()}', header = true)
""")
print("Created table: train_run_events")

Created table: train_run_events


### 3.2 `train_master`

In [4]:
con.execute(f"""
    CREATE OR REPLACE TABLE train_master AS
    SELECT *
    FROM read_csv('{(DATA_DIR / "train_master.csv").as_posix()}', header = true)
""")
print("Created table: train_master")

Created table: train_master


### 3.3 `station_master`

In [5]:
con.execute(f"""
    CREATE OR REPLACE TABLE station_master AS
    SELECT *
    FROM read_csv('{(DATA_DIR / "station_master.csv").as_posix()}', header = true)
""")
print("Created table: station_master")

Created table: station_master


## 4. Inspect table schemas

Inferred types are not a contract. Confirm the date and timestamp columns did not fall back to `VARCHAR`.

In [6]:
con.sql("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'main'
    ORDER BY table_name
""").show()

┌────────────────────────┐
│       table_name       │
│        varchar         │
├────────────────────────┤
│ station_master         │
│ station_master_clean   │
│ train_master           │
│ train_master_clean     │
│ train_run_events       │
│ train_run_events_clean │
└────────────────────────┘



In [7]:
con.sql("DESCRIBE train_run_events").show()

┌───────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│        column_name        │ column_type │  null   │   key   │ default │  extra  │
│          varchar          │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ service_id                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ service_date              │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ train_id                  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ train_name                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ station_sequence          │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ station_id                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ station_name              │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ scheduled_arrival         │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NU

In [8]:
con.sql("DESCRIBE train_master").show()

┌─────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│     column_name     │ column_type │  null   │   key   │ default │  extra  │
│       varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ train_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ train_name          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ origin_station      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ destination_station │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ service_type        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ route_note          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘



In [9]:
con.sql("DESCRIBE station_master").show()

┌──────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name  │ column_type │  null   │   key   │ default │  extra  │
│   varchar    │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ station_id   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ station_name │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ region       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└──────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘



## 5. Verify the load

### 5.1 Row counts

Each table must hold as many rows as its CSV has data lines. A mismatch stops the notebook.

In [10]:
con.sql("""
    SELECT 'train_run_events' AS table_name, COUNT(*) AS row_count FROM train_run_events
    UNION ALL
    SELECT 'train_master', COUNT(*) FROM train_master
    UNION ALL
    SELECT 'station_master', COUNT(*) FROM station_master
""").show()

┌──────────────────┬───────────┐
│    table_name    │ row_count │
│     varchar      │   int64   │
├──────────────────┼───────────┤
│ train_run_events │      2460 │
│ train_master     │        10 │
│ station_master   │        20 │
└──────────────────┴───────────┘



In [11]:
for table in RAW_TABLES:
    # fine as long as no field has an embedded newline
    with open(DATA_DIR / f"{table}.csv", newline="", encoding="utf-8") as f:
        csv_rows = sum(1 for _ in f) - 1  # minus the header
    table_rows = con.sql(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    assert table_rows == csv_rows, f"{table}: {table_rows} rows loaded, {csv_rows} in CSV"
    print(f"OK  {table}: {table_rows:,} rows match the CSV")

OK  train_run_events: 2,460 rows match the CSV
OK  train_master: 10 rows match the CSV
OK  station_master: 20 rows match the CSV


### 5.2 Key checks

Pass: `distinct_keys` equals `row_count` and `null_keys` is 0.

In [12]:
con.sql("""
    SELECT
        'train_run_events' AS table_name,
        'service_id, station_sequence' AS key_columns,
        COUNT(*) AS row_count,
        COUNT(DISTINCT (service_id, station_sequence)) AS distinct_keys,
        COUNT(*) FILTER (WHERE service_id IS NULL OR station_sequence IS NULL) AS null_keys
    FROM train_run_events
    UNION ALL
    SELECT 'train_master', 'train_id', COUNT(*), COUNT(DISTINCT train_id),
        COUNT(*) FILTER (WHERE train_id IS NULL)
    FROM train_master
    UNION ALL
    SELECT 'station_master', 'station_id', COUNT(*), COUNT(DISTINCT station_id),
        COUNT(*) FILTER (WHERE station_id IS NULL)
    FROM station_master
""").show()

┌──────────────────┬──────────────────────────────┬───────────┬───────────────┬───────────┐
│    table_name    │         key_columns          │ row_count │ distinct_keys │ null_keys │
│     varchar      │           varchar            │   int64   │     int64     │   int64   │
├──────────────────┼──────────────────────────────┼───────────┼───────────────┼───────────┤
│ train_run_events │ service_id, station_sequence │      2460 │          2460 │         0 │
│ train_master     │ train_id                     │        10 │            10 │         0 │
│ station_master   │ station_id                   │        20 │            20 │         0 │
└──────────────────┴──────────────────────────────┴───────────┴───────────────┴───────────┘



### 5.3 Coverage of the events table

Date range and distinct runs, trains and stations. Baseline for later notebooks.

In [13]:
con.sql("""
    SELECT
        MIN(service_date) AS first_service_date,
        MAX(service_date) AS last_service_date,
        COUNT(DISTINCT service_id) AS train_runs,
        COUNT(DISTINCT train_id) AS trains,
        COUNT(DISTINCT station_id) AS stations
    FROM train_run_events
""").show()

┌────────────────────┬───────────────────┬────────────┬────────┬──────────┐
│ first_service_date │ last_service_date │ train_runs │ trains │ stations │
│        date        │       date        │   int64    │ int64  │  int64   │
├────────────────────┼───────────────────┼────────────┼────────┼──────────┤
│ 2026-08-01         │ 2026-09-14        │        426 │     10 │       19 │
└────────────────────┴───────────────────┴────────────┴────────┴──────────┘



## 6. Preview the tables

In [14]:
con.sql("SELECT * FROM train_run_events ORDER BY service_id, station_sequence LIMIT 5").show(max_width=200)

┌────────────┬──────────────┬──────────┬─────────────────┬──────────────────┬───┬─────────────────────┬─────────────────────┬───────────────────────────┬────────────────────────┬─────────────────────┐
│ service_id │ service_date │ train_id │   train_name    │ station_sequence │ … │  actual_departure   │ departure_delay_min │ simulated_disruption_flag │ simulated_disruption_… │     data_status     │
│  varchar   │     date     │ varchar  │     varchar     │      int64       │ … │      timestamp      │        int64        │          boolean          │        varchar         │       varchar       │
├────────────┼──────────────┼──────────┼─────────────────┼──────────────────┼───┼─────────────────────┼─────────────────────┼───────────────────────────┼────────────────────────┼─────────────────────┤
│ RUN000001  │ 2026-08-01   │ TR001    │ Subarna Express │                1 │ … │ 2026-08-01 07:02:00 │                   2 │ false                     │ none                   │ SYNTHETIC_SIMULAT

In [15]:
con.sql("SELECT * FROM train_master ORDER BY train_id LIMIT 5").show(max_width=200)

┌──────────┬───────────────────┬────────────────┬─────────────────────┬──────────────┬───────────────────────────────────────────────────────┐
│ train_id │    train_name     │ origin_station │ destination_station │ service_type │                      route_note                       │
│ varchar  │      varchar      │    varchar     │       varchar       │   varchar    │                        varchar                        │
├──────────┼───────────────────┼────────────────┼─────────────────────┼──────────────┼───────────────────────────────────────────────────────┤
│ TR001    │ Subarna Express   │ Dhaka          │ Chattogram          │ Intercity    │ Illustrative route; not an official current timetable │
│ TR002    │ Mohanagar Express │ Dhaka          │ Chattogram          │ Intercity    │ Illustrative route; not an official current timetable │
│ TR003    │ Tista Express     │ Dhaka          │ Jamalpur            │ Intercity    │ Illustrative route; not an official current timetable │

In [16]:
con.sql("SELECT * FROM station_master ORDER BY station_id LIMIT 5").show(max_width=200)

┌────────────┬──────────────┬─────────┐
│ station_id │ station_name │ region  │
│  varchar   │   varchar    │ varchar │
├────────────┼──────────────┼─────────┤
│ ST001      │ Dhaka        │ Central │
│ ST002      │ Biman Bandar │ Central │
│ ST003      │ Joydebpur    │ Central │
│ ST004      │ Tangail      │ West    │
│ ST005      │ Ibrahimabad  │ West    │
└────────────┴──────────────┴─────────┘



## 7. Close the connection

Releases the write lock so other notebooks can open the file.

In [17]:
# release the lock so the other notebooks can open the db
con.close()
print("Connection closed.")

Connection closed.
